## 准备数据

In [36]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [37]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [38]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 100], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.truncated_normal([100, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [39]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)

    # 手动
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)
    # 使用 Adam 优化器自动更新可以大幅提升准确率
    # optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [40]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.4438248 ; accuracy 0.0964
epoch 1 : loss 2.4307353 ; accuracy 0.09793333
epoch 2 : loss 2.4183433 ; accuracy 0.09956667
epoch 3 : loss 2.406568 ; accuracy 0.10125
epoch 4 : loss 2.3953416 ; accuracy 0.10351667
epoch 5 : loss 2.384601 ; accuracy 0.106233336
epoch 6 : loss 2.3742964 ; accuracy 0.10886667
epoch 7 : loss 2.3643837 ; accuracy 0.11145
epoch 8 : loss 2.354826 ; accuracy 0.11478333
epoch 9 : loss 2.3455884 ; accuracy 0.1185
epoch 10 : loss 2.3366425 ; accuracy 0.12201667
epoch 11 : loss 2.3279605 ; accuracy 0.1256
epoch 12 : loss 2.3195174 ; accuracy 0.12963334
epoch 13 : loss 2.3112972 ; accuracy 0.13388333
epoch 14 : loss 2.3032792 ; accuracy 0.13871667
epoch 15 : loss 2.2954488 ; accuracy 0.14315
epoch 16 : loss 2.2877905 ; accuracy 0.1478
epoch 17 : loss 2.280291 ; accuracy 0.15233333
epoch 18 : loss 2.2729387 ; accuracy 0.15745
epoch 19 : loss 2.2657225 ; accuracy 0.16281667
epoch 20 : loss 2.2586331 ; accuracy 0.16785
epoch 21 : loss 2.2516608 ; accuracy

In [41]:
# optimizer.apply_gradients(zip(grads, trainable_vars))
# 使用 Adam 优化器自动更新可以大幅提升准确率，输出如下：
# epoch 0 : loss 2.554716 ; accuracy 0.1002
# epoch 1 : loss 2.4288542 ; accuracy 0.10975
# epoch 2 : loss 2.3261173 ; accuracy 0.13348334
# epoch 3 : loss 2.2391734 ; accuracy 0.17495
# epoch 4 : loss 2.1627297 ; accuracy 0.22818333
# epoch 5 : loss 2.0931594 ; accuracy 0.29318333
# epoch 6 : loss 2.0278924 ; accuracy 0.36315
# epoch 7 : loss 1.9651313 ; accuracy 0.42663333
# epoch 8 : loss 1.9036995 ; accuracy 0.48495
# epoch 9 : loss 1.8427957 ; accuracy 0.5356333
# epoch 10 : loss 1.7819829 ; accuracy 0.5765167
# epoch 11 : loss 1.7210934 ; accuracy 0.6077167
# epoch 12 : loss 1.6601158 ; accuracy 0.6322
# epoch 13 : loss 1.5991409 ; accuracy 0.6515833
# epoch 14 : loss 1.538361 ; accuracy 0.6696333
# epoch 15 : loss 1.4780607 ; accuracy 0.68635
# epoch 16 : loss 1.4185349 ; accuracy 0.7032167
# epoch 17 : loss 1.3600608 ; accuracy 0.71928334
# epoch 18 : loss 1.3029393 ; accuracy 0.73373336
# epoch 19 : loss 1.2475145 ; accuracy 0.74728334
# epoch 20 : loss 1.1940862 ; accuracy 0.75916666
# epoch 21 : loss 1.1429302 ; accuracy 0.76928335
# epoch 22 : loss 1.0942779 ; accuracy 0.77825
# epoch 23 : loss 1.0482771 ; accuracy 0.78603333
# epoch 24 : loss 1.0049565 ; accuracy 0.79285
# epoch 25 : loss 0.96425927 ; accuracy 0.79971665
# epoch 26 : loss 0.9260555 ; accuracy 0.80613333
# epoch 27 : loss 0.8901578 ; accuracy 0.81196666
# epoch 28 : loss 0.8564085 ; accuracy 0.81771666
# epoch 29 : loss 0.8247171 ; accuracy 0.8229667
# epoch 30 : loss 0.79504186 ; accuracy 0.82836664
# epoch 31 : loss 0.7673457 ; accuracy 0.83308333
# epoch 32 : loss 0.74156314 ; accuracy 0.83706665
# epoch 33 : loss 0.71760684 ; accuracy 0.8413
# epoch 34 : loss 0.6953215 ; accuracy 0.8437833
# epoch 35 : loss 0.6745594 ; accuracy 0.8466167
# epoch 36 : loss 0.655177 ; accuracy 0.84935
# epoch 37 : loss 0.63706917 ; accuracy 0.8516
# epoch 38 : loss 0.6201328 ; accuracy 0.8531833
# epoch 39 : loss 0.6042801 ; accuracy 0.85538334
# epoch 40 : loss 0.58943045 ; accuracy 0.8571333
# epoch 41 : loss 0.5754981 ; accuracy 0.85906667
# epoch 42 : loss 0.5624103 ; accuracy 0.8609333
# epoch 43 : loss 0.5500971 ; accuracy 0.86321664
# epoch 44 : loss 0.53851444 ; accuracy 0.86548334
# epoch 45 : loss 0.5276191 ; accuracy 0.8674167
# epoch 46 : loss 0.5173628 ; accuracy 0.86918336
# epoch 47 : loss 0.5076843 ; accuracy 0.87093335
# epoch 48 : loss 0.49851933 ; accuracy 0.8728167
# epoch 49 : loss 0.4898222 ; accuracy 0.87448335
# test loss 0.46176308 ; accuracy 0.8831